In [ ]:
import json
import os
from datetime import datetime, timedelta

os.environ['PROJ_IGNORE_CELESTIAL_BODY'] = 'YES'

import copernicusmarine
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
import cartopy.crs as ccrs
import cartopy.feature as cfeature
import numpy as np
from shapely.geometry import shape
from opendrift.models.oceandrift import OceanDrift
from opendrift.readers.reader_netCDF_CF_generic import Reader
import geopandas as gpd

Load the oil slick GeoJSON and extract the MultiPolygon geometry.

In [ ]:
# # North Sea (near land)
# # https://cerulean.skytruth.org/slicks/4603914
# slick_id = 4603914
# observation_timestamp = datetime(2025, 12, 20, 17, 9, 40)

# North Sea (cross shaped slick)
# https://cerulean.skytruth.org/slicks/4247543
slick_id = 4247543
observation_timestamp = datetime(2025, 7, 21, 17, 26, 59)

# Near Taiwan
# https://cerulean.skytruth.org/slicks/4245242
# slick_id = 4245242
# observation_timestamp = datetime(2025, 7, 20, 10, 1, 23)

# # Near Indonesia
# #https://cerulean.skytruth.org/slicks/4167052
# slick_id = 4167052
# observation_timestamp = datetime(2025, 6, 13, 21, 36, 15)

slick_file = f'data/slick-{slick_id}.geo.json'
with open(slick_file) as f:
    slick_geojson = json.load(f)

slick_all = shape(slick_geojson['features'][0]['geometry'])
slick_polygons = list(slick_all.geoms) if slick_all.geom_type == 'MultiPolygon' else [slick_all]
print(f'Using all {len(slick_polygons)} polygons')

# Save with explicit CRS for OpenDrift shapefile overlay
slick_gdf = gpd.read_file(slick_file).set_crs('EPSG:4326')
slick_overlay = f'outputs/slick-{slick_id}.gpkg'
slick_gdf.to_file(slick_overlay, driver='GPKG')

In [ ]:
duration_hours = 24
num_particles = 1000

Get an Xarray dataset from the copernicusmarine client.

In [ ]:
ds = copernicusmarine.open_dataset(
    dataset_id='cmems_mod_glo_phy_anfc_merged-uv_PT1H-i',
    chunk_size_limit=0,
)
print(ds)

Create OpenDrift readers with different current component mappings.

In [ ]:
reader_default = Reader(ds, name='CMEMS default')

reader_tides = Reader(ds, standard_name_mapping={
    'utide': 'x_sea_water_velocity',
    'vtide': 'y_sea_water_velocity',
}, name='Tides only')

reader_stokes = Reader(ds, standard_name_mapping={
    'vsdx': 'x_sea_water_velocity',
    'vsdy': 'y_sea_water_velocity',
}, name='Stokes only')

reader_total = Reader(ds, standard_name_mapping={
    'utotal': 'x_sea_water_velocity',
    'vtotal': 'y_sea_water_velocity',
}, name='Total current')

Run forward and backward simulations for each current component.

In [ ]:
reader_combined = reader_default + reader_tides + reader_stokes

directions = {
    'forward': 3600,
    'backward': -3600,
}

total_area = sum(p.area for p in slick_polygons)

simulations = {}
for dname, time_step in directions.items():
    o = OceanDrift()
    o.add_reader(reader_combined, variables=['x_sea_water_velocity', 'y_sea_water_velocity'])
    for polygon in slick_polygons:
        lons, lats = polygon.exterior.coords.xy
        o.seed_within_polygon(
            lons=lons, lats=lats,
            number=max(1, int(polygon.area / total_area * num_particles)),
            time=observation_timestamp,
        )
    o.run(duration=timedelta(hours=duration_hours), time_step=time_step)
    simulations[dname] = o

Trajectory overview with start and end positions.

In [ ]:
direction_colors = {
    'forward': 'tab:blue',
    'backward': 'tab:red',
}

fig, ax = plt.subplots(figsize=(12, 8), subplot_kw={'projection': ccrs.PlateCarree()})
ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, zorder=2)
gl = ax.gridlines(draw_labels=True, alpha=0.3)
gl.top_labels = False
gl.right_labels = False

# Compute extent from all trajectories
all_lons = []
all_lats = []
for o in simulations.values():
    all_lons.append(o.result.lon.values.ravel())
    all_lats.append(o.result.lat.values.ravel())
all_lons = np.concatenate(all_lons)
all_lats = np.concatenate(all_lats)
all_lons = all_lons[~np.isnan(all_lons)]
all_lats = all_lats[~np.isnan(all_lats)]
pad = 0.1
ax.set_extent([all_lons.min() - pad, all_lons.max() + pad,
               all_lats.min() - pad, all_lats.max() + pad])

handles = []
for dname, o in simulations.items():
    color = direction_colors[dname]
    lon = o.result.lon.values
    lat = o.result.lat.values

    # Plot trajectories
    step = max(1, lon.shape[0] // 200)
    for j in range(0, lon.shape[0], step):
        mask = ~np.isnan(lon[j])
        ax.plot(lon[j][mask], lat[j][mask], color=color,
                alpha=0.3, linewidth=0.5, transform=ccrs.PlateCarree())

    # Plot start positions
    ax.scatter(lon[:, 0], lat[:, 0], s=6, color=color,
               alpha=0.6, transform=ccrs.PlateCarree(), zorder=4)
    # Plot end positions
    end_idx = np.array([np.max(np.where(~np.isnan(lon[j]))) for j in range(lon.shape[0])])
    end_lons = np.array([lon[j, end_idx[j]] for j in range(lon.shape[0])])
    end_lats = np.array([lat[j, end_idx[j]] for j in range(lon.shape[0])])
    ax.scatter(end_lons, end_lats, s=6, marker='x', color=color,
               alpha=0.6, transform=ccrs.PlateCarree(), zorder=4)

    handles.append(mlines.Line2D([], [], color=color, linewidth=1.5,
                                 label=f'{dname.capitalize()} {duration_hours}h'))

# Plot observed slick outline
for polygon in slick_polygons:
    x, y = polygon.exterior.coords.xy
    ax.plot(x, y, color='black', linewidth=1, transform=ccrs.PlateCarree(), zorder=5)
handles.append(mlines.Line2D([], [], color='black', linewidth=1, label='Observed slick'))

ax.legend(handles=handles, loc='best')
ax.set_title(f'Slick {slick_id} | Forward & backward drift {duration_hours}h')
plt.tight_layout()
fig_file = f'outputs/drift_trajectories-{slick_id}.png'
fig.savefig(fig_file, dpi=150, bbox_inches='tight')
print(f'Saved {fig_file}')
plt.show()

Combined animation showing forward (blue) and backward (red) particle trajectories.

In [ ]:
from matplotlib import animation

fwd = simulations['forward']
bwd = simulations['backward']

fwd_lon = fwd.result.lon.values  # shape: (n_particles, n_times)
fwd_lat = fwd.result.lat.values
bwd_lon = bwd.result.lon.values
bwd_lat = bwd.result.lat.values
n_frames = min(fwd_lon.shape[1], bwd_lon.shape[1])

fwd_times = fwd.result.time.values
bwd_times = bwd.result.time.values

# Compute map extent from all trajectory data
all_lon = np.concatenate([fwd_lon.ravel(), bwd_lon.ravel()])
all_lat = np.concatenate([fwd_lat.ravel(), bwd_lat.ravel()])
all_lon = all_lon[~np.isnan(all_lon)]
all_lat = all_lat[~np.isnan(all_lat)]
pad = 0.1
extent = [all_lon.min() - pad, all_lon.max() + pad,
          all_lat.min() - pad, all_lat.max() + pad]

fig, ax = plt.subplots(figsize=(10, 8), subplot_kw={'projection': ccrs.PlateCarree()})
ax.set_extent(extent)
ax.add_feature(cfeature.LAND, facecolor='lightgray', zorder=1)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, zorder=2)

# Static: observed slick outline
for polygon in slick_polygons:
    x, y = polygon.exterior.coords.xy
    ax.plot(x, y, color='black', linewidth=1, transform=ccrs.PlateCarree(), zorder=5)

# Scatter plots for particles (updated each frame)
fwd_scatter = ax.scatter([], [], s=4, color='tab:blue', alpha=0.5, transform=ccrs.PlateCarree(), zorder=4)
bwd_scatter = ax.scatter([], [], s=4, color='tab:red', alpha=0.5, transform=ccrs.PlateCarree(), zorder=4)

# Trail lines (accumulated each frame)
trail_lines_fwd = []
trail_lines_bwd = []

title = ax.set_title('')

handles = [
    mlines.Line2D([], [], color='black', linewidth=1, label='Observed slick'),
    mlines.Line2D([], [], color='tab:blue', marker='o', markersize=4, linestyle='None', label='Forward'),
    mlines.Line2D([], [], color='tab:red', marker='o', markersize=4, linestyle='None', label='Backward'),
]
ax.legend(handles=handles, loc='upper left')

def update(frame):
    # Update forward particles
    fx = fwd_lon[:, frame]
    fy = fwd_lat[:, frame]
    fmask = ~np.isnan(fx)
    fwd_scatter.set_offsets(np.c_[fx[fmask], fy[fmask]])

    # Update backward particles
    bx = bwd_lon[:, frame]
    by = bwd_lat[:, frame]
    bmask = ~np.isnan(bx)
    bwd_scatter.set_offsets(np.c_[bx[bmask], by[bmask]])

    # Add trail segments
    if frame > 0:
        step = max(1, fwd_lon.shape[0] // 150)
        for j in range(0, fwd_lon.shape[0], step):
            if not np.isnan(fwd_lon[j, frame-1]) and not np.isnan(fwd_lon[j, frame]):
                ln, = ax.plot([fwd_lon[j, frame-1], fwd_lon[j, frame]],
                              [fwd_lat[j, frame-1], fwd_lat[j, frame]],
                              color='tab:blue', alpha=0.15, linewidth=0.5,
                              transform=ccrs.PlateCarree(), zorder=3)
                trail_lines_fwd.append(ln)
            if not np.isnan(bwd_lon[j, frame-1]) and not np.isnan(bwd_lon[j, frame]):
                ln, = ax.plot([bwd_lon[j, frame-1], bwd_lon[j, frame]],
                              [bwd_lat[j, frame-1], bwd_lat[j, frame]],
                              color='tab:red', alpha=0.15, linewidth=0.5,
                              transform=ccrs.PlateCarree(), zorder=3)
                trail_lines_bwd.append(ln)

    fwd_t = np.datetime_as_string(fwd_times[frame], unit='h')
    bwd_t = np.datetime_as_string(bwd_times[frame], unit='h')
    title.set_text(f'Slick {slick_id} | {duration_hours}h drift | Fwd: {fwd_t} | Bwd: {bwd_t}')
    return fwd_scatter, bwd_scatter, title

anim = animation.FuncAnimation(fig, update, frames=n_frames, interval=300)
gif_file = f'outputs/drift_combined-{slick_id}.gif'
anim.save(gif_file, writer='pillow', fps=4)
plt.close(fig)
print(f'Saved {gif_file} ({n_frames} frames)')

In [ ]:
from IPython.display import Image, display
display(Image(filename=f'outputs/drift_combined-{slick_id}.gif'))